# Estimator Examples

This notebook demonstrates the common workflow for ACM estimator classes. It builds a mock catalog, prepares a shared backend, selects an estimator by statistic name, and calls `compute` with statistic-specific arguments.

## Setup

The example uses a periodic Lagrangian mock from `mockfactory` and a `JaxpowerBackend`. The only registered estimator here is the bispectrum, but the pattern is the part to reuse for other statistics.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
%config InlineBackend.figure_format = "retina"

from cosmoprimo.fiducial import DESI
from mockfactory import LagrangianLinearMock

from acm import setup_logging
from acm.estimators.galaxy_clustering.backends.jaxpower import JaxpowerBackend
from acm.estimators.galaxy_clustering.bispectrum import BispectrumMultipoles

setup_logging()


## Build A Mock Catalog

The helper returns Cartesian redshift-space positions in a periodic box. RSD is enabled by default and uses the same line of sight passed to the estimator.

In [ ]:
def make_lagrangian_mock(
    boxsize=500.0,
    nbar=8e-4,
    nmesh=256,
    bias=2.0,
    redshift=0.5,
    seed=42,
    rsd=True,
    los="z",
):
    """Return positions from a small mockfactory Lagrangian mock."""
    cosmo = DESI()
    power = (
        cosmo.get_fourier(engine="eisenstein_hu", set_engine=False)
        .pk_interpolator()
        .to_1d(z=redshift)
    )
    mock = LagrangianLinearMock(
        power,
        nmesh=nmesh,
        boxsize=boxsize,
        boxcenter=0.0,
        seed=seed,
        unitary_amplitude=False,
    )
    mock.set_real_delta_field(bias=bias - 1.0)
    mock.set_analytic_selection_function(nbar=nbar)
    mock.poisson_sample(seed=seed + 1)
    if rsd:
        f = cosmo.growth_rate(redshift)
        mock.set_rsd(f=f, los=los)
    return np.asarray(mock.to_catalog()["Position"]), boxsize


## Estimator Helpers

These local helpers are intentionally small. Extend the registry and argument helper as new estimator classes are added.

In [ ]:
def get_estimator(stat_name):
    """Return the estimator class associated with a statistic name."""
    estimators = {
        "bispectrum": BispectrumMultipoles,
    }
    return estimators[stat_name]


def get_compute_args(stat_name, basis="scoccimarro"):
    """Return compact compute arguments for the example statistic."""
    if stat_name != "bispectrum":
        raise ValueError(f"No compute arguments registered for {stat_name!r}.")

    return {
        "basis": basis,
        "edges": {"min": 0.02, "max": 0.20, "step": 0.02},
        "buffer_size": 30,
        "resampler": "tsc",
        "interlacing": 3,
        "compensate": True,
    }


def summarize_bispectrum(result, ell=None, nrows=5):
    """Print a small summary of a Mesh3SpectrumPoles result."""
    ell = result.ells[0] if ell is None else ell
    pole = result.get(ell)
    k = np.asarray(pole.coords("k"))

    print(type(result))
    print(f"basis: {result.basis}")
    print(f"ells: {result.ells}")
    print(f"number of bins: {len(k)}")
    print("first k coordinates:")
    print(k[:nrows])


## Shared Backend

The backend owns the particle field and mesh configuration. Several estimators can reuse the same backend instance.

In [ ]:
los = "z"
data_positions, boxsize = make_lagrangian_mock(boxsize=500.0, los=los)

backend = JaxpowerBackend(data_positions, boxsize=boxsize, cellsize=5.0)
backend.set_density_contrast(
    resampler="tsc",
    interlacing=3,
    compensate=True,
)

print(f"Number of mock particles: {len(data_positions)}")
print(f"Box size: {boxsize:.1f} Mpc/h")
print(f"Mesh size: {backend.meshsize}")


## Generic Estimator Workflow

Select the statistic, retrieve its estimator class, instantiate it with the shared backend, and call `compute`. Statistic-specific details stay inside `get_compute_args`.

In [ ]:
stat_name = "bispectrum"
cls = get_estimator(stat_name)
estimator = cls(backend=backend, data_positions=data_positions)

result = estimator.compute(los=los, **get_compute_args(stat_name))


## Inspect The Result

The concrete result type depends on the statistic. For the current bispectrum example, the result is an `lsstypes.Mesh3SpectrumPoles` object.

In [ ]:
summarize_bispectrum(result)
